# RAG Application

![Simple RAG](../../images/simple_rag.png)

在本笔记中，我们将搭建一个简单的 RAG 应用，并在学习 LangSmith 的过程中使用它。

RAG（检索增强生成）是一种流行的技术，它为语言学习模型 (LLM) 提供相关文档，使其能够更好地回答用户的问题。

在本例中，我们将索引一些 LangSmith 文档！

LangSmith 可以轻松追踪任何 LLM 应用，无需 LangChain！

### Setup

Make sure you set your environment variables, including your OpenAI API key.

In [ ]:
# You can set them inline!
# import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Simple RAG application

In [2]:
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio
import os
from utils import get_vector_db_retriever

MODEL_PROVIDER = "qwen"
MODEL_NAME = "qwen3-max"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- Returns documents fetched from a vectorstore based on the user's question
"""
@traceable(run_type="chain", metadata={"vectordb": "sklearn"})
def retrieve_documents(question: str):
    return retriever.invoke(question)

"""
generate_response
- Calls `call_openai` to generate a model response after formatting inputs
"""
@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

"""
call_openai
- Returns the chat completion output from OpenAI
"""
@traceable(run_type="llm", metadata={"model_name": MODEL_NAME, "model_provider": MODEL_PROVIDER})
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

"""
langsmith_rag
- Calls `retrieve_documents` to fetch documents
- Calls `generate_response` to generate a response based on the fetched documents
- Returns the model response
"""
@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

完整文件路径: C:\Users\Kevin\AppData\Local\Temp\union.parquet


当运行带有 `@traceable` 装饰器函数时，会在函数调用时创建一个运行树（模拟嵌套函数调用）。
- 检测这是否是一个根运行，即一个新的跟踪，还是已经存在一个父运行，而这个新运行实际是一个嵌套函数调用。
- 如果发现被调用的函数带有 `@traceable` ，并且父函数也带有 `@traceable`，会将这个新运行插入到父运行中。

metadata 元数据传递方式：
- **静态传递**
    - `@traceable(metadata={"model_name": MODEL_NAME, "model_provider": MODEL_PROVIDER})`
- **动态传递**
    - `langsmith_rag(question, langsmith_extra={"metadata": {"website": "www.google.com"}})`
 
run_type 运行类型：
- **LLM：** LLM 运行涉及调用 LLM 或聊天模型
- **Retriever：** 检索器运行涉及从某个外部来源获取补充文档或数据
- **Tool：** 工具运行用于当我们使用一个模型来创建工具调用作为输出时
- **Chain： Chain是默认运行类型，** 它只是表示我们应用程序中的任意执行步骤
- **Prompt：** Prompt运行通常涉及从模板创建提示
- **Parser：** 解析器运行用于将一些非结构化输出解析为结构化模式

In [4]:
question = "What is LangSmith used for?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"website": "www.google.com"}})
print(ai_answer)

LangSmith is used for developing, debugging, and deploying LLM applications. It provides tools for tracing requests, evaluating outputs, testing prompts, and managing deployments. It works with or without LangChain’s open-source libraries and supports local prototyping and production monitoring.


### Let's take a look in LangSmith!

进入到 LangSmith 后台查看项目

<img src="../../images/m0_1.png">

<img src="../../images/m0_2.png">
